# Customer Analytics: CLV Regression & Churn Prediction
**Retail analytics project - code companion to the project report**

This notebook reproduces every step of the report:

| Report section | Notebook section |
|---|---|
| Q2 EDA | 2 |
| Q3 Missing data (median imputation) | 3 |
| Q1 Hypothesis tests (Pearson) | 4 |
| Q6 Multicollinearity | 5 |
| Q4 / Q5 OLS regression for CLV + F-test | 6 |
| Q7 / Q8 Logistic regression + odds ratios | 7 |
| Q9 Confusion matrix & metrics | 8 |
| Q10 Improvements (FE, selection, SMOTE/class weights, non-linear models) | 9 |
| Business insights (numbers) | 10 |
| Report vs notebook check | 11 |

**How to use:** upload your customer CSV (10,000 rows x 32 columns) when prompted, or put it at `/content/customer_data.csv`.
If no file is found, a *synthetic demo dataset* is generated so you can see the notebook run end-to-end (its results will NOT match the report).

## 0. Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, LogisticRegression, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict, cross_val_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, precision_recall_curve)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

## 1. Configuration & data loading
Edit the column names below if your CSV uses different headers.

In [ ]:
DATA_PATH = "/content/customer_data.csv"   # change if needed

# ---- column names (edit to match your file) ----
CLV   = "CLV"
CHURN = "Churn_Flag"
LR_PREDICTORS = ["Age", "Income", "Credit_Score", "Monthly_Spend", "Num_Transactions",
                 "Website_Visits", "Customer_Satisfaction", "Loyalty_Points",
                 "Tenure_Years", "Return_Rate", "Product_Rating"]          # 11 OLS predictors
LOGIT_PREDICTORS = [c for c in LR_PREDICTORS if c != "Credit_Score"]       # 10 predictors used in report's logit table
HYP_PREDICTORS = ["Income", "Monthly_Spend", "Loyalty_Points"]             # Q1 hypothesis tests
ALPHA = 0.05

# The report's models use all 10,000 rows (F-test df = 9,988), i.e. the targets were median-imputed too.
# True  -> replicate the report.   False -> drop rows with a missing target instead (methodologically cleaner).
IMPUTE_TARGETS = True

In [ ]:
def make_synthetic(n=10_000, seed=RANDOM_STATE):
    """Demo data with the same structure/skew as the report. Targets are unrelated to predictors."""
    r = np.random.default_rng(seed)
    d = pd.DataFrame({
        "Age": r.integers(18, 70, n).astype(float),
        "Income": r.lognormal(10.5, 0.9, n),
        "Credit_Score": r.normal(650, 80, n),
        "Monthly_Spend": r.lognormal(7.2, 0.9, n),
        "Num_Transactions": r.poisson(25, n).astype(float),
        "Website_Visits": r.poisson(30, n).astype(float),
        "Customer_Satisfaction": r.normal(3.5, 0.8, n),
        "Loyalty_Points": r.lognormal(6.5, 0.9, n),
        "Tenure_Years": r.uniform(0, 15, n),
        "Return_Rate": r.uniform(0, 0.4, n),
        "Product_Rating": r.normal(4.1, 0.63, n),
        CLV: r.lognormal(9.6, 0.95, n),
        CHURN: (r.random(n) < 0.63).astype(float),
    })
    for i in range(1, 20):                       # filler columns -> 32 variables in total
        d[f"Feature_{i:02d}"] = r.normal(size=n)
    mask = r.random(d.shape) < 0.08               # ~8% missing everywhere
    return d.mask(mask)

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Upload your customer data file (.csv / .xlsx)...")
        up = files.upload()
        if up:
            DATA_PATH = list(up.keys())[0]
    except ImportError:
        pass

if os.path.exists(DATA_PATH):
    df_raw = pd.read_excel(DATA_PATH) if DATA_PATH.lower().endswith((".xlsx", ".xls")) else pd.read_csv(DATA_PATH)
    print(f"Loaded {DATA_PATH}")
    USING_SYNTHETIC = False
else:
    df_raw = make_synthetic()
    print("No data file found -> using SYNTHETIC demo data (results will not match the report).")
    USING_SYNTHETIC = True

needed = set(LR_PREDICTORS + [CLV, CHURN])
missing_cols = sorted(needed - set(df_raw.columns))
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}\nAvailable columns: {list(df_raw.columns)}\n"
                   "-> edit the column-name variables in the configuration cell.")
print("Shape:", df_raw.shape)
df_raw.head()

## 2. Exploratory data analysis (Q2)

In [ ]:
key_vars = ["Age", "Income", "Monthly_Spend", CLV, "Customer_Satisfaction", "Loyalty_Points", "Product_Rating", CHURN]
key_vars = [c for c in key_vars if c in df_raw.columns]
desc = df_raw[key_vars].agg(["count", "mean", "std", "min", "median", "max", "skew"]).T
desc.columns = ["N valid", "Mean", "Std Dev", "Min", "Median", "Max", "Skew"]
print("Table 1: Descriptive statistics")
display(desc)

p_churn = df_raw[CHURN].mean()
print(f"\nChurn rate among valid rows: {p_churn:.2%}")
print(f"Income  skew = {df_raw['Income'].skew():.2f} (mean {df_raw['Income'].mean():,.0f} vs median {df_raw['Income'].median():,.0f})")
print(f"CLV     skew = {df_raw[CLV].skew():.2f} (mean {df_raw[CLV].mean():,.0f} vs median {df_raw[CLV].median():,.0f})")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
sns.histplot(df_raw["Age"].dropna(), bins=25, color="#2E86AB", ax=ax[0]); ax[0].set_title("Age Distribution")
sns.histplot(df_raw[CLV].dropna(), bins=50, color="#E63946", ax=ax[1]); ax[1].set_title("CLV Distribution")
sns.histplot(df_raw["Income"].dropna(), bins=50, color="#6A4C93", ax=ax[2]); ax[2].set_title("Income Distribution")
vc = df_raw[CHURN].value_counts().sort_index()
ax[3].bar(["Retained (0)", "Churned (1)"], vc.values, color=["#3AAF6E", "#E63946"])
for i, v in enumerate(vc.values):
    ax[3].text(i, v, f"{v:,}\n({v/vc.sum():.1%})", ha="center", va="bottom", fontweight="bold")
ax[3].set_title("Churn Distribution")
plt.tight_layout(); plt.show()

## 3. Missing data & median imputation (Q3)

In [ ]:
miss = (df_raw.isna().mean() * 100).sort_values(ascending=False).rename("% missing")
print(f"Variables with missing values: {(miss > 0).sum()} of {df_raw.shape[1]}")
print(f"Range of missingness: {miss.min():.1f}% - {miss.max():.1f}%")
display(miss.head(10).to_frame())

df = df_raw.copy()
if not IMPUTE_TARGETS:
    df = df.dropna(subset=[CLV, CHURN])
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
cat_cols = df.select_dtypes(exclude=np.number).columns
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode().iloc[0])
print(f"\nRemaining nulls after imputation: {int(df.isna().sum().sum())}   |   rows used for modelling: {len(df):,}")
print(f"Churn rate after imputation: {df[CHURN].mean():.2%}")

## 4. Hypothesis tests - Pearson correlation with CLV (Q1)
H0: rho = 0 (no linear relationship) vs H1: rho != 0, two-tailed, alpha = 0.05.

In [ ]:
rows = []
for v in HYP_PREDICTORS:
    r, p = stats.pearsonr(df[v], df[CLV])
    rows.append({"Predictor": v, "H0 / H1": "rho=0 / rho!=0", "Pearson r": r, "p-value": p,
                 "Decision": "Reject H0" if p < ALPHA else "Fail to reject H0"})
hyp = pd.DataFrame(rows)
print("Table 2: Hypothesis tests")
display(hyp)

## 5. Multicollinearity - correlation matrix & VIF (Q6)

In [ ]:
corr = df[LR_PREDICTORS + [CLV]].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0, linewidths=0.4, annot_kws={"size": 7})
plt.title("Pearson Correlation Heatmap (multicollinearity check)"); plt.tight_layout(); plt.show()

pred_corr = corr.loc[LR_PREDICTORS, LR_PREDICTORS].abs().values.copy()
np.fill_diagonal(pred_corr, 0)
print(f"Largest absolute pairwise correlation among predictors: {pred_corr.max():.3f}  (threshold 0.70)")

def compute_vif(X):
    out = {}
    for c in X.columns:
        others = X.drop(columns=c)
        r2 = LinearRegression().fit(others, X[c]).score(others, X[c])
        out[c] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(out, name="VIF")

vif = compute_vif(df[LR_PREDICTORS])
display(vif.to_frame())

## 6. Multivariate linear regression for CLV + F-test (Q4, Q5)

In [ ]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

X_ols = sm.add_constant(df[LR_PREDICTORS])
ols = sm.OLS(df[CLV], X_ols).fit()

ci = ols.conf_int()
ols_tbl = pd.DataFrame({
    "Coeff (beta)": ols.params, "Std Err": ols.bse, "t-stat": ols.tvalues, "p-value": ols.pvalues,
    "CI Low": ci[0], "CI High": ci[1], "Sig.": [stars(p) for p in ols.pvalues],
    "VIF": vif.reindex(ols.params.index)
})
print("Table 3: OLS coefficients (*p<0.10, **p<0.05, ***p<0.01)")
display(ols_tbl)

n_obs, k = int(ols.nobs), len(LR_PREDICTORS)
print(f"\nR-squared        : {ols.rsquared:.4f}")
print(f"Adjusted R2      : {ols.rsquared_adj:.4f}")
print(f"F-test           : F({k}, {n_obs-k-1}) = {ols.fvalue:.3f},  p = {ols.f_pvalue:.4f}")
print(f"Log-likelihood   : {ols.llf:,.1f}")
print(f"AIC              : {ols.aic:,.1f}")
print(f"Condition number : {ols.condition_number:,.3g}  (large -> predictors on very different scales)")
if ols.f_pvalue < ALPHA:
    print("\nDecision: reject H0 -> the model is jointly significant.")
else:
    print(f"\nDecision: p = {ols.f_pvalue:.3f} > {ALPHA} -> fail to reject H0; the predictors jointly do not explain CLV.")

plt.figure(figsize=(7, 4.5))
c = ols.params.drop("const")
colors = ["#E63946" if ols.pvalues[i] < 0.10 else "#2E86AB" for i in c.index]
plt.barh(c.index, c.values, color=colors); plt.axvline(0, color="k", lw=0.8)
plt.title("OLS coefficients (red = p<0.10)"); plt.tight_layout(); plt.show()

## 7. Logistic regression for churn + odds ratios (Q7, Q8)

In [ ]:
X_log = sm.add_constant(df[LOGIT_PREDICTORS])
y = df[CHURN].astype(int)
try:
    logit = sm.Logit(y, X_log).fit(disp=0, maxiter=200)
except Exception:                       # fall back if Newton-Raphson struggles with unscaled inputs
    logit = sm.Logit(y, X_log).fit(disp=0, method="bfgs", maxiter=2000)

ci_l = logit.conf_int()
logit_tbl = pd.DataFrame({
    "Coeff (beta)": logit.params, "Std Err": logit.bse, "z-stat": logit.tvalues,
    "Odds Ratio": np.exp(logit.params), "OR CI Low": np.exp(ci_l[0]), "OR CI High": np.exp(ci_l[1]),
    "p-value": logit.pvalues, "Sig.": [stars(p) for p in logit.pvalues],
    "Direction": np.where(logit.params > 0, "up churn", "down churn")
})
print("Table 4: Logistic regression")
display(logit_tbl)

print(f"\nMcFadden pseudo-R2 : {logit.prsquared:.4f}")
print(f"Log-likelihood     : {logit.llf:,.2f}")
print(f"AIC                : {logit.aic:,.2f}")
sig = logit_tbl[(logit_tbl['p-value'] < 0.10) & (logit_tbl.index != 'const')]
print("Significant predictors (p<0.10):", list(sig.index) if len(sig) else "none")

In [ ]:
# Forest plot of odds ratios
t = logit_tbl.drop(index="const")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(t["Odds Ratio"], range(len(t)),
            xerr=[t["Odds Ratio"] - t["OR CI Low"], t["OR CI High"] - t["Odds Ratio"]], fmt="o", color="#2E86AB", capsize=3)
ax.axvline(1, color="red", ls="--"); ax.set_yticks(range(len(t))); ax.set_yticklabels(t.index)
ax.set_xlabel("Odds ratio (95% CI)"); ax.set_title("Odds ratios for churn"); plt.tight_layout(); plt.show()

## 8. Classification performance at the 0.5 threshold (Q9)

In [ ]:
y_prob = np.asarray(logit.predict(X_log))
y_pred = (y_prob >= 0.5).astype(int)
cm = confusion_matrix(y, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

metrics = {
    "Accuracy": accuracy_score(y, y_pred),
    "Precision": precision_score(y, y_pred, zero_division=0),
    "Recall": recall_score(y, y_pred, zero_division=0),
    "F1-score": f1_score(y, y_pred, zero_division=0),
}
print(f"TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}")
display(pd.Series(metrics, name="Value").to_frame())
print(f"Share of observations predicted as churned: {y_pred.mean():.2%}")
if y_pred.min() == y_pred.max():
    print("-> The model predicts a single class for every customer (degenerate classifier caused by class imbalance).")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax[0],
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["Act 0", "Act 1"])
ax[0].set_title("Confusion Matrix (threshold = 0.5)")
bars = ax[1].bar(list(metrics.keys()), list(metrics.values()), color=["#2E86AB", "#F4A261", "#3AAF6E", "#9B5DE5"])
for b in bars: ax[1].text(b.get_x() + b.get_width()/2, b.get_height(), f"{b.get_height():.4f}", ha="center", va="bottom", fontweight="bold")
ax[1].set_ylim(0, 1.1); ax[1].set_title("Classification Metrics"); plt.tight_layout(); plt.show()

## 9. Improvements (Q10)
These go beyond the report's baseline: standardisation, class weights, threshold tuning, cross-validation, non-linear models, feature engineering and selection.
> If your real data truly contains no signal, expect AUC near 0.50 and R2 near 0 here as well. That is itself a valid finding.

In [ ]:
# 9a. Standardised logistic regression with class weights, stratified 10-fold CV (instead of the degenerate default)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
Xc = df[LOGIT_PREDICTORS]

bal_lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
p_cv = cross_val_predict(bal_lr, Xc, y, cv=skf, method="predict_proba")[:, 1]

prec, rec, thr = precision_recall_curve(y, p_cv)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
best_thr = thr[np.argmax(f1s[:-1])]

for name, t in [("threshold 0.50", 0.5), (f"F1-optimal threshold {best_thr:.2f}", best_thr)]:
    pr = (p_cv >= t).astype(int)
    print(f"{name:32s} acc={accuracy_score(y, pr):.3f}  prec={precision_score(y, pr, zero_division=0):.3f}  "
          f"recall={recall_score(y, pr):.3f}  F1={f1_score(y, pr):.3f}")
print(f"\nCross-validated ROC-AUC (balanced logistic): {roc_auc_score(y, p_cv):.4f}")
print("Confusion matrix @0.5 (rows = actual 0/1):\n", confusion_matrix(y, (p_cv >= 0.5).astype(int)))

In [ ]:
# 9b. Non-linear models for churn (10-fold CV, ROC-AUC)
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced",
                                            n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}
res = {n: cross_val_score(m, Xc, y, cv=skf, scoring="roc_auc", n_jobs=-1) for n, m in models.items()}
res["Balanced Logistic"] = cross_val_score(bal_lr, Xc, y, cv=skf, scoring="roc_auc")
display(pd.DataFrame({k: {"Mean AUC": v.mean(), "Std": v.std()} for k, v in res.items()}).T)

In [ ]:
# 9c. Feature engineering: log-transform skewed variables + interaction terms, re-test CLV model with 10-fold CV
fe = df.copy()
for c in ["Income", "Monthly_Spend", "Loyalty_Points"]:
    fe[f"log_{c}"] = np.log1p(fe[c])
fe["Income_x_Loyalty"] = fe["log_Income"] * fe["log_Loyalty_Points"]
fe["Spend_x_Tenure"] = fe["log_Monthly_Spend"] * fe["Tenure_Years"]
fe["Spend_per_Txn"] = fe["Monthly_Spend"] / fe["Num_Transactions"].replace(0, np.nan)
fe["Spend_per_Txn"] = fe["Spend_per_Txn"].fillna(fe["Spend_per_Txn"].median())

fe_feats = LR_PREDICTORS + ["log_Income", "log_Monthly_Spend", "log_Loyalty_Points",
                            "Income_x_Loyalty", "Spend_x_Tenure", "Spend_per_Txn"]
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
y_log_clv = np.log1p(df[CLV])

def cv_r2(model, X, target):
    return cross_val_score(model, X, target, cv=kf, scoring="r2").mean()

out = pd.Series({
    "Baseline linear (raw CLV)": cv_r2(make_pipeline(StandardScaler(), LinearRegression()), df[LR_PREDICTORS], df[CLV]),
    "Linear + engineered feats (log CLV)": cv_r2(make_pipeline(StandardScaler(), LinearRegression()), fe[fe_feats], y_log_clv),
    "Random Forest (log CLV)": cv_r2(RandomForestRegressor(n_estimators=150, min_samples_leaf=10, n_jobs=-1,
                                                           random_state=RANDOM_STATE), fe[fe_feats], y_log_clv),
}, name="CV R2")
display(out.to_frame())

In [ ]:
# 9d. Feature selection: LASSO for CLV, RFE for churn
Xs = StandardScaler().fit_transform(fe[fe_feats])
lasso = LassoCV(cv=5, random_state=RANDOM_STATE).fit(Xs, y_log_clv)
coef = pd.Series(lasso.coef_, index=fe_feats)
print(f"LASSO alpha = {lasso.alpha_:.4g}; features kept: {(coef != 0).sum()} of {len(coef)}")
display(coef[coef != 0].sort_values(key=abs, ascending=False).to_frame("LASSO coef"))

rfe = RFE(LogisticRegression(class_weight="balanced", max_iter=1000), n_features_to_select=5)
rfe.fit(StandardScaler().fit_transform(Xc), y)
print("\nRFE - top 5 churn predictors:", list(Xc.columns[rfe.support_]))

# Optional: if you build RFM / digital-engagement features, add them to `fe_feats` and re-run 9b-9d.

## 10. Business-insight numbers

In [ ]:
beta_pr = ols.params["Product_Rating"]
n_rated = int(df_raw["Product_Rating"].notna().sum())
uplift_per_cust = beta_pr * 0.5
print(f"1) Product Rating: beta = GBP {beta_pr:,.0f} per rating point (p = {ols.pvalues['Product_Rating']:.3f})")
print(f"   +0.5 rating point -> GBP {uplift_per_cust:,.0f} per customer x {n_rated:,} rated customers "
      f"= GBP {uplift_per_cust * n_rated / 1e6:,.2f}M potential CLV  (indicative only - p > 0.05)")

n_valid_churn = int(df_raw[CHURN].notna().sum())
n_churned = int(df_raw[CHURN].sum())
print(f"\n2) Churn: {n_churned:,} of {n_valid_churn:,} valid customers churned ({n_churned / n_valid_churn:.1%})")
print(f"   5 percentage-point churn reduction on {n_valid_churn:,} customers -> ~{0.05 * n_valid_churn:,.0f} extra customers retained")
print(f"   5% RELATIVE reduction in the {n_churned:,} churners             -> ~{0.05 * n_churned:,.0f} extra customers retained")

print(f"\n3) Model signal: R2 (CLV) = {ols.rsquared:.4f}, pseudo-R2 (churn) = {logit.prsquared:.4f}")

## 11. Report vs notebook check
Compare these numbers with the report (they should match when you run on the original dataset).

In [ ]:
report_vals = {
    "OLS R-squared": (0.0012, ols.rsquared),
    "OLS Adj. R-squared": (0.0001, ols.rsquared_adj),
    "F-statistic": (1.125, ols.fvalue),
    "F-test p-value": (0.336, ols.f_pvalue),
    "OLS AIC": (225_800, ols.aic),
    "Product Rating coeff": (702.09, ols.params["Product_Rating"]),
    "Product Rating p-value": (0.083, ols.pvalues["Product_Rating"]),
    "McFadden pseudo-R2": (0.0003, logit.prsquared),
    "Logit AIC": (13_191.58, logit.aic),
    "Accuracy": (0.6305, metrics["Accuracy"]),
    "Precision": (0.6305, metrics["Precision"]),
    "Recall": (1.0, metrics["Recall"]),
    "F1-score": (0.773, metrics["F1-score"]),
}
chk = pd.DataFrame(report_vals, index=["Report", "Notebook"]).T
chk["Match (5% tol.)"] = np.isclose(chk["Report"], chk["Notebook"], rtol=0.05, atol=1e-3)
display(chk)
if USING_SYNTHETIC:
    print("Note: synthetic demo data in use - differences are expected.")